In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_neutron_295K_278464_unopt_rPBE_SEDC_magres.magres"


This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file

*Make sure you change the nucleus information like Q value and nucleus (I am not sure how to do this in an efficient way)*

Shiva Agarwal

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'O'      # nucleus for which parameters are wanted
atom_label = 4      # site for which parameters wanted
Q = -0.0256         #electric quadrupole moment for nucleus in barn

In [7]:
for atom in atoms.species(nucleus):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-48.05747428 163.36822913 -77.0563951 ]
 [166.49989977  89.13706204  26.75251019]
 [  6.2018781  -21.96825493 -39.32765365]]

17O2 sigma:
 [[ -48.05747428 -163.36822913   77.0563951 ]
 [-166.49989977   89.13706204   26.75251019]
 [  -6.2018781   -21.96825493  -39.32765365]]

17O3 sigma:
 [[-48.05747428 163.36822913  77.0563951 ]
 [166.49989977  89.13706204 -26.75251019]
 [ -6.2018781   21.96825493 -39.32765365]]

17O4 sigma:
 [[ -48.05747428 -163.36822913  -77.0563951 ]
 [-166.49989977   89.13706204  -26.75251019]
 [   6.2018781    21.96825493  -39.32765365]]

17O5 sigma:
 [[ -35.27799775  204.22751995   54.52126487]
 [ 176.61542246  129.7960294   -81.67725922]
 [  15.9451448   -43.89730276 -148.88778293]]

17O6 sigma:
 [[ -35.27799775 -204.22751995  -54.52126487]
 [-176.61542246  129.7960294   -81.67725922]
 [ -15.9451448   -43.89730276 -148.88778293]]

17O7 sigma:
 [[ -35.27799775  204.22751995  -54.52126487]
 [ 176.61542246  129.7960294    81.67725922]
 [ -15.9451448 

In [8]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.6297323103941554

17O2 sigma:
 6.629732310394153

17O3 sigma:
 6.6297323103941945

17O4 sigma:
 6.629732310394204

17O5 sigma:
 8.255453591556629

17O6 sigma:
 8.25545359155664

17O7 sigma:
 8.255453591556662

17O8 sigma:
 8.255453591556691



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-4.262  0.87  -1.258]
 [ 0.87  -3.64   1.75 ]
 [-1.258  1.75   7.902]]

CS Tensor:
 [[ -35.278  204.228   54.521]
 [ 176.615  129.796  -81.677]
 [  15.945  -43.897 -148.888]]

CS isotropic Tensor:
 [[-18.123   0.      0.   ]
 [  0.    -18.123   0.   ]
 [  0.      0.    -18.123]]

CS symmetric Tensor:
 [[ -35.278  190.421   35.233]
 [ 190.421  129.796  -62.787]
 [  35.233  -62.787 -148.888]]

CS antisymmetric Tensor:
 [[  0.     13.806  19.288]
 [-13.806   0.    -18.89 ]
 [-19.288  18.89    0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 8.26190538 -5.19756701 -3.06433836] 

 Unsorted Eigenvectors:
 [[-0.08944903 -0.77704684  0.62305463]
 [ 0.13847396  0.60978438  0.78037681]
 [ 0.98631832 -0.15608079 -0.05305613]] 

Sorted Eigenvalues: 
 [-3.06433836 -5.19756701  8.26190538] 

Sorted Eigenvectors: 
 [[ 0.62305463 -0.77704684 -0.08944903]
 [ 0.78037681  0.60978438  0.13847396]
 [-0.05305613 -0.15608079  0.98631832]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 257.56946076  -92.12615959 -219.81305245] 

 Unsorted Eigenvectors:
 [[-0.53617548 -0.60860447 -0.58490721]
 [-0.83998836  0.31633462  0.44085369]
 [ 0.08327913 -0.72769019  0.68083153]] 

Sorted Eigenvalues: 
 [ -92.12615959 -219.81305245  257.56946076] 

Sorted Eigenvectors: 
 [[-0.60860447 -0.58490721 -0.53617548]
 [ 0.31633462  0.44085369 -0.83998836]
 [-0.72769019  0.68083153  0.08327913]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.0643383628935363 -5.197567013783974 8.26190537667776
CSA Tensor Components δyy, δxx, δzz: 
 -92.12615958793887 -219.8130524503018 257.5694607592878


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 17O5: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   8.26191  |
+--------------+------------+
| etaq         |   0.258201 |
+--------------+------------+
| iso_cs (ppm) | -18.1233   |
+--------------+------------+
| csa (ppm)    | 275.693    |
+--------------+------------+
| etas         |   0.463149 |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/O_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.77704684  0.62305463 -0.08944903]
 [ 0.60978438  0.78037681  0.13847396]
 [-0.15608079 -0.05305613  0.98631832]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
18.774300650751496 9.488634200545095 57.13904104430934 

Direction cosine csa: 

[[-0.58490721 -0.60860447 -0.53617548]
 [ 0.44085369  0.31633462 -0.83998836]
 [ 0.68083153 -0.72769019  0.08327913]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-46.90541136807328 85.22292480718656 -57.44938080323269 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -38.283268427907856 chi: 89.21023307293052 xi: -83.76613688883052 

